# RAG: With a Framework, Then From Scratch

**Retrieval-Augmented Generation (RAG)** answers questions using facts the model wasn't trained on — by retrieving relevant documents at query time and stuffing them into the prompt as context.

Both parts of this notebook answer the same question over the same tiny knowledge base, so you can compare the two approaches directly:

1. **Part 1 — With a Framework:** [`llama_index`](https://www.llamaindex.ai/)'s `VectorStoreIndex` and `query_engine` handle embedding, indexing, retrieval, and generation behind a few lines of code.
2. **Part 2 — From Scratch:** The same pipeline built by hand with just the OpenAI SDK and NumPy — embed, compare via cosine similarity, retrieve, and generate — so the mechanism a framework hides is fully visible.

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai openai numpy python-dotenv

---
## Step 2: Set Up Environment Variables

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

---
## Part 1: RAG with a Framework (`llama_index`)
`Settings.llm` and `Settings.embed_model` configure which models power generation and embedding globally. `VectorStoreIndex.from_documents` embeds and indexes the knowledge base in one call, and `as_query_engine()` wires up retrieval + generation automatically.

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.openai import OpenAI as LlamaIndexOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# LLM: The "Brain"
Settings.llm = LlamaIndexOpenAI(model="gpt-4o-mini")

# EMBEDDING: The "Search"
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    embed_batch_size=100  # Sends multiple items in one go to save time
)

# Simple Array Knowledge Base
kb_array = [
    "The 2025 Tokyo Protocol mandates a global coal phase-out by 2035.",
    "NASA's Artemis IV landed a pressurized rover on Mars in January 2026.",
    "The March 2026 AI Act requires digital watermarking on all AI media.",
    "The SF 49ers won Super Bowl LX in February 2026 against the Chiefs."
]

# Build and Query
documents = [Document(text=item) for item in kb_array]

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

print("\n--- Response ---")
print(query_engine.query("What are the new AI rules for 2026?"))

---
## Part 2: RAG From Scratch
No framework here — just the OpenAI SDK and NumPy, so the actual mechanism (embed, compare, retrieve, generate) is visible instead of hidden behind `llama_index`'s `VectorStoreIndex`/`query_engine` abstractions.

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI()


def embed(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([item.embedding for item in response.data])


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_norm = a / np.linalg.norm(a, axis=-1, keepdims=True)
    b_norm = b / np.linalg.norm(b)
    return a_norm @ b_norm


def query(question: str, top_k: int = 2) -> str:
    # RETRIEVE: embed the knowledge base and the question, then rank
    # documents by cosine similarity to find the most relevant ones.
    doc_embeddings = embed(kb_array)
    query_embedding = embed([question])[0]

    similarities = cosine_similarity(doc_embeddings, query_embedding)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    retrieved_docs = [kb_array[i] for i in top_indices]

    # AUGMENT + GENERATE: stuff the retrieved context into the prompt and
    # ask the model to answer using only that context.
    context = "\n".join(retrieved_docs)
    prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer using only the context above."

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


print("\n--- Response ---")
print(query("What are the new AI rules for 2026?"))